# Step 29 — cross-platform, use A: transfer the endotypes to the RNA sequencing studies

**Data types: RNA_array → bulk_RNA_seq, scRNA_seq pseudobulk.** **Reads:** `step13_model.rds`,
`step13_site_*`, `step23_GSE232381.rds`, `step25_GSE135779.rds`. **Writes:** `step29_transfer.rds`.

The model found on the array sites is sent to each RNA sequencing study as parameters: the gene list,
and the centroids **divided by the pooled discovery SD of each gene**, which puts them in standardised
units. The receiving study standardises its own samples the same way (each gene centred on its own
SLE patients' mean and divided by their SD), labels each sample by the centroid it correlates with
best, and returns counts.

**The gate:** assignment is refused when the study measures fewer than 70% of the model's genes
(`assign_external` in `src/endotypes.R`, from the reference scaffold).

**Test the transfer on the array first.** The array's own validation patients, relabelled by this
standardised route, should mostly keep the labels step 13 gave them. If they did not, the route
itself would be at fault.

In [1]:
source("../src/paths.R")
source("../src/endotypes.R")
source("../src/federation.R")
start_log("29")
model <- readRDS(art("step13_model.rds"))
cz <- sweep(model$centroids, 2, model$sd, "/")                  # centroids in SD units
standardise_sle <- function(E, sle) {                            # samples x genes, on own SLE patients
  E <- E[intersect(model$genes, rownames(E)), , drop = FALSE]
  t((E - rowMeans(E[, sle, drop = FALSE])) / apply(E[, sle, drop = FALSE], 1, sd))
}

## Check on the array's validation patients

In [2]:
chk <- sapply(SITES, function(s) {
  d <- readRDS(site_file("13", s)); m <- d$meta
  v <- m[m$split == "validation", ]; v <- v[order(v$subject, v$visit), ]; j <- rownames(v)[!duplicated(v$subject)]
  z <- standardise_sle(d$E[, j], rep(TRUE, length(j)))
  lab <- assign_external(z, cz)$endotype
  send(c(patients = length(j), agree = sum(as.character(lab) == as.character(m[j, "endotype"]))), s, "transfer check counts", length(j))
})
chk
c(agreement = round(sum(chk["agree", ]) / sum(chk["patients", ]), 3))

,A,B,C
patients,16,16,16
agree,16,16,11


agreement 
    0.896

## GSE232381 (bulk_RNA_seq)

In [3]:
b <- readRDS(art("step23_GSE232381.rds"))
zb <- standardise_sle(b$E, rep(TRUE, ncol(b$E)))
tb <- assign_external(zb, cz)
c(coverage = round(attr(tb, "coverage"), 3))
tab_b <- send(table(ln_activity = b$meta$ln_activity, endotype = tb$endotype), "GSE232381", "endotype counts", ncol(b$E))
tab_b
fisher.test(tab_b)$p.value

coverage 
   0.776

           endotype
ln_activity E1 E2
   inactive  4  2
   active    7  3

[1] 1

## GSE135779 (scRNA_seq pseudobulk)

In [4]:
p <- readRDS(art("step25_GSE135779.rds"))
zp <- standardise_sle(p$E, p$meta$disease == "SLE")
tp <- tryCatch(assign_external(zp, cz), error = function(e) { message("REFUSED: ", conditionMessage(e)); NULL })
is.null(tp)

REFUSED: only 68% of centroid genes present; refusing to assign



[1] TRUE

The gate refuses GSE135779: PBMC pseudobulk measures too few of the model's genes, mostly because
the granulocyte genes are missing (step 26). We do not lower the threshold to get a label. Step 30
takes the other route: clustering together on the genes all studies share.

In [5]:
saveRDS(list(check = chk, GSE232381 = cbind(b$meta, tb), GSE232381_table = tab_b,
             GSE135779 = if (is.null(tp)) "refused: gene coverage below 70%" else tp), art("step29_transfer.rds"))

## Findings

- **The transfer route works on the array itself.** 43 of 48 validation patients (90%) keep their
  step 13 label when relabelled through standardised units. The misses come from site C, where
  centring on 16 patients' own mean shifts the reference.
- **GSE232381 (bulk_RNA_seq), coverage 78%:** 11 samples are E1 and 5 are E2. Endotype is not related
  to nephritis activity (Fisher p = 1.0). With 16 patients this can only rule out a very strong
  association.
- **GSE135779 (scRNA_seq pseudobulk), coverage 68%: refused.** The model cannot be applied to PBMC
  because the genes that define E2 are not measured there. This is the gate doing its job.